# Notebook 04 — Failure Mode Analysis

Investigates systematic failure patterns in RAG pipeline outputs:
- Failure mode taxonomy and definitions
- Per-mode frequency and distribution analysis
- Low-scoring run deep dives (trace inspection)
- Retrieval failure cases: weak retrieval vs. missing citation
- Abstention quality: when the system should and shouldn't abstain
- Recommendations for addressing each failure type

**Prerequisites:** Run `python scripts/run_baseline.py` first.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().absolute().parent / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
matplotlib.rcParams['figure.dpi'] = 120
%matplotlib inline

from llm_evals_lab.config import load_config
from llm_evals_lab.observability.run_store import RunStore

cfg = load_config()
run_store = RunStore(cfg.runs_dir(), cfg.tables_dir())
df = run_store.to_dataframe()

if df.empty:
    print('No run data. Run scripts/run_baseline.py first.')
else:
    print(f'Loaded {len(df)} run records')

## 1. Failure Mode Taxonomy

In [ ]:
taxonomy = {
    'unsupported_answer': 'Answer makes claims not grounded in retrieved context (groundedness < 0.30)',
    'weak_retrieval': 'Relevant documents were not retrieved (context_recall < 0.30)',
    'missing_citation': 'Answerable question answered without citing expected documents (citation_coverage < 0.20)',
    'incorrect_abstention': 'System abstained on a question that has a valid answer in the corpus',
    'overconfident_answer': 'System answered a question with no relevant information available',
    'incomplete_answer': 'Answer is too brief (< 10 words) for a non-trivial question',
    'none': 'No failure modes detected — run passed all quality checks',
}
for mode, definition in taxonomy.items():
    print(f'• {mode}:')
    print(f'  {definition}')
    print()

## 2. Failure Mode Frequency

In [ ]:
if not df.empty and 'failure_modes' in df.columns:
    all_modes = []
    for fm in df['failure_modes'].dropna():
        all_modes.extend([m.strip() for m in str(fm).split('|') if m.strip()])
    mode_counts = Counter(all_modes)
    total = len(df)

    fm_df = pd.DataFrame(
        [(k, v, round(v/total*100, 1)) for k, v in mode_counts.most_common()],
        columns=['failure_mode', 'count', 'rate_%']
    )
    print('Failure mode distribution:')
    display(fm_df)

    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ['#55A868' if k == 'none' else '#E24A33' for k in fm_df['failure_mode']]
    bars = ax.barh(fm_df['failure_mode'], fm_df['count'], color=colors)
    ax.set_xlabel('Count')
    ax.set_title(f'Failure Mode Frequency (n={total} runs)')
    for bar, rate in zip(bars, fm_df['rate_%']):
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                f'{rate}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig('../results/figures/04_failure_modes.png', dpi=150, bbox_inches='tight')
    plt.show()

## 3. Failure Modes by Experiment

In [ ]:
if not df.empty and 'failure_modes' in df.columns and 'experiment_id' in df.columns:
    from llm_evals_lab.experiments.compare import ExperimentComparison
    comp = ExperimentComparison(df)
    fm_pivot = comp.failure_mode_summary()
    if not fm_pivot.empty:
        print('Failure modes by experiment:')
        display(fm_pivot)

## 4. Hardest Examples (Lowest Scores)

In [ ]:
if not df.empty and 'overall_score' in df.columns:
    bottom = df.nsmallest(8, 'overall_score')[[
        'example_id', 'experiment_id', 'query', 'overall_score',
        'groundedness', 'hit_at_k', 'failure_modes'
    ]].fillna('N/A')
    print('8 lowest-scoring runs:')
    display(bottom)

## 5. Deep Dive: Trace Inspection for a Failing Run

In [ ]:
if not df.empty and 'overall_score' in df.columns:
    # Load the trace for the worst-scoring run
    worst_row = df.nsmallest(1, 'overall_score').iloc[0]
    worst_run_id = worst_row['run_id']
    record = run_store.load(worst_run_id)

    if record:
        print(f'=== WORST RUN TRACE ===')
        print(f'Run ID:      {record.run_id}')
        print(f'Experiment:  {record.experiment_id}')
        print(f'Query:       {record.query}')
        print()
        print(f'Retrieved chunks ({len(record.retrieved_chunks)}):')
        for c in record.retrieved_chunks:
            print(f'  [{c.rank}] {c.chunk_id} score={c.score:.3f}')
        print()
        if record.generated_answer:
            print(f'Answer: {record.generated_answer.answer_text}')
            print(f'Abstained: {record.generated_answer.abstained}')
        print()
        if record.metrics:
            m = record.metrics
            print(f'Metrics:')
            print(f'  Overall score:    {m.answer.overall_score:.3f}')
            print(f'  Groundedness:     {m.answer.groundedness_score:.3f}')
            print(f'  Citation cov:     {m.answer.citation_coverage_score:.3f}')
            print(f'  Context recall:   {m.retrieval.context_recall:.3f}')
            print(f'  Hallucination:    {m.answer.hallucination_risk_score:.3f}')
            print(f'  Failure modes:    {[fm.value for fm in m.failure_modes]}')

## 6. Recommendations

Based on the failure analysis above, here are actionable recommendations:

| Failure Mode | Root Cause | Recommendation |
|---|---|---|
| `weak_retrieval` | TF-IDF misses semantic matches | Upgrade to sentence-transformers embeddings |
| `missing_citation` | Local generator doesn't cite by default | Switch to `grounded` prompt strategy |
| `incorrect_abstention` | Abstention threshold too high | Lower `abstain_threshold` in generator config |
| `overconfident_answer` | Generator answers despite low overlap | Raise `abstain_threshold`; filter by retrieval score |
| `unsupported_answer` | Retrieved chunks are off-topic | Increase `top_k`; add re-ranking step |
| `incomplete_answer` | Low-overlap query triggers minimal extraction | Add minimum answer length enforcement |
